# Implementación del algoritmo de optimización heurístico ABC (Artificial Bee Colony)

## Parámetros de control del algoritmo

- **SN**: número de fuentes de alimento (soluciones).
- **MCN**: número máximo de ciclos de iteración.
- **limit**: número máximo de intentos de mejora de una solución.
- **f**: función objetivo.

In [1]:
# Librerías necesarias
from __future__ import annotations
from typing import Callable
import numpy as np
import math


## Implementación principal

In [ ]:
class ArtificialBeeColony:
    # Atributos
    limit: int
    sn: int
    mcn: int
    f: Callable
    seed: int # Semilla de aleatorización.
    history: dict # Para guardar el historial

    def __init__(self, limit: int, sn: int, mcn: int, f: Callable, seed: int) -> None:
        self.limit = limit
        self.sn = sn
        self.mcn = mcn
        self.f = f
        self.seed = seed
        self.history = {}

    def execute(self, restrictions: list[tuple]):

        if self.sn < 2:
            raise Exception("Debe haber mínimo dos soluciones.")
            
        self.history = {}
        rng = np.random.default_rng(self.seed)
        solutions: list = [None]*self.sn
        trial : list = [0]*self.sn # Contador de intentos

        # Inicialización de los vectores de solución.
        rows = []
        for i in range(self.sn):
            solutions[i] = [self.generate_random_solution(rng, lower, upper) for lower, upper in restrictions]
            rows.append({"Solución": i+1, 
                        "Componentes": solutions[i].copy(), 
                        "Valor en función objetivo": self.f(*solutions[i]),
                        })

        self.history["inicializacion"] = rows

        mat_solutions = np.array(solutions)
        init_values = np.array([self.f(*s) for s in mat_solutions])
        best_index = np.argmin(init_values)
        best_solution = mat_solutions[best_index].copy()
        best_value = init_values[best_index]

        # Inicio del algoritmo
        for cyc in range(self.mcn):
            # Fase de abejas obreras
            rows = []
            replaced = [False]*self.sn
            for i in range(self.sn):
                
                new_solution, k, j, phi = self.generate_neighbor_solution(rng, mat_solutions, restrictions, i)
                rows.append({
                    "Ciclo": cyc + 1,
                    f"Solución Nueva": i+1,
                    "k": k,
                    "j": j,
                    "phi": phi,
                    "Componentes": new_solution.copy(),
                })
                
                if self.f(*new_solution) < self.f(*mat_solutions[i]):
                    mat_solutions[i] = new_solution
                    trial[i] = 0
                    replaced[i] = True 
                else:
                    trial[i] += 1

            self.history.setdefault("candidatas_fase_obreras", []).extend(rows)

            # Intermedio, se contruye la tabla de reemplazos de esta fase
            rows = []
            for i in range(self.sn):
                rows.append({
                    "Ciclo": cyc + 1,
                    "Solución": i + 1,
                    "Reemplazada": replaced[i],
                    "Componentes": mat_solutions[i].copy(),
                    "Valor en función objetivo": self.f(*mat_solutions[i]),
                    "Trial": trial[i],
                })

            self.history.setdefault("reemplazadas_fase_obreras", []).extend(rows)

            # Fase de abejas observadoras
            fitness = [self.fitness(mat_solutions, i) for i in range(self.sn)]
            max_fitness = max(fitness)
            p = [self.calculate_probability(fitness[i], max_fitness) for i in range(self.sn)]

            i = 0
            t = 0
            rows = []
            replaced = [False]*self.sn
            while t < self.sn:
                r = rng.uniform(0, 1)
                record = {"Ciclo": cyc + 1, "Iteración (Solución evaluada)": i + 1, "Probabilidad": p[i], "r": r, "r < p": r < p[i]}
                if r < p[i]:
                    t += 1
                    new_solution, k, j, phi = self.generate_neighbor_solution(rng, mat_solutions, restrictions, i)
                    record.update({"Nuevo candidato": new_solution.copy(), 
                              "Valor en función objetivo": self.f(*new_solution)})
                    if self.f(*new_solution) < self.f(*mat_solutions[i]):
                        mat_solutions[i] = new_solution
                        trial[i] = 0
                        record.update({"Reemplazada": True, "Trial": 0})
                        replaced[i] = True
                    else:
                        trial[i] += 1
                        record.update({"Reemplazada": False, "Trial": trial[i]})

                record.update({"Trial": trial[i]})
                rows.append(record)

                i = (i + 1) % self.sn

            self.history.setdefault("fase_abejas_observadoras", []).extend(rows)

            # Intermedio, se contruye la tabla de reemplazos de esta fase
            rows = []
            for i in range(self.sn):
                rows.append({
                    "Ciclo": cyc + 1,
                    "Solución": i + 1,
                    "Reemplazada": replaced[i],
                    "Componentes": mat_solutions[i].copy(),
                    "Valor en función objetivo": self.f(*mat_solutions[i]),
                    "Trial": trial[i],
                })

            self.history.setdefault("fase_abejas_observadoras_resumen", []).extend(rows)

            # Fase de abejas exploradoras

            values = np.array([self.f(*solution) for solution in mat_solutions])
            best_index = np.argmin(values)
            if best_value is None or values[best_index] < best_value:
                best_solution = mat_solutions[best_index].copy()
                best_value = values[best_index]


            replaced = [False]*self.sn
            for i, num_trial in enumerate(trial):
                if num_trial > self.limit:
                    mat_solutions[i] = np.array([
                        self.generate_random_solution(rng, lower, upper) for lower, upper in restrictions])
                    trial[i] = 0
                    replaced[i] = True

            # Registro del resultado de la fase exploradora
            rows = []
            for i in range(self.sn):
                rows.append({
                    "Ciclo": cyc + 1,
                    "Solución": i + 1,
                    "Reemplazada": replaced[i],
                    "Componentes": mat_solutions[i].copy(),
                    "Valor en función objetivo": self.f(*mat_solutions[i]),
                    "Trial": trial[i],
                })

            self.history.setdefault("fase_abejas_exploradoras", []).extend(rows)

        return best_solution, best_value, self.history

    def generate_random_solution(self, rng, lower_bound: float, upper_bound : float) -> float:
        return lower_bound + rng.uniform(0, 1)*(upper_bound - lower_bound)

    def generate_neighbor_solution(self, rng, solutions, restrictions: list[tuple], i: int) -> tuple:
    
        j = rng.choice(np.arange(len(solutions[i])))
        restriction_lower, restriction_upper = restrictions[j]

        new_solution = solutions[i].copy()

        phi = None
        k = None
        new_value = None
        while True:
            phi = rng.uniform(-1, 1)
            indexes = np.arange(len(solutions))
            valid_indexes = indexes[indexes != i]
            k = rng.choice(valid_indexes)
            new_value = solutions[i, j] + phi*(solutions[i, j] - solutions[k, j])
            if restriction_lower <= new_value <= restriction_upper:
                break
            elif restriction_lower == restriction_upper:
                new_value = restriction_lower
                break

        new_solution[j] = new_value

        return new_solution, k, j, phi
        
    def fitness(self, solutions, i : int) -> float:
        f = self.f(*solutions[i])
        if f >= 0:
            return 1/(1 + f)
        else:
            return 1 + abs(f)

    def calculate_probability(self, fitness_i: float, max_fitness: float) -> float:
        return 0.1 + 0.9*(fitness_i / max_fitness)
        


In [3]:
from abc_visualizer import show_abc 

# Función objetivo
f = lambda x1, x2, x3, x4, x5: x1**2 + x2**2 + x3**2 + x4**2 + x5**2
# Instancia del algoritmo
abc = ArtificialBeeColony(2, 5, 60, f, 42)
# Restricciones de cada componente
restrictions = [(-5.12, 5.12), (-5.12, 5.12), (-5.12, 5.12), (-5.12, 5.12), (-5.12, 5.12)]
# Ejecución del algoritmo
best_solution, best_value, history = abc.execute(restrictions)
# Representación gráfica del algoritmo
show_abc(history, abc.f, restrictions)

In [7]:
from abc_visualizer import show_abc
# Función objetivo
# f = lambda x1: x1**2 + 10*(math.sin(x1)**2)
f = lambda x1: 418.9829 - x1*math.sin(abs(x1)**0.5)
# Restricciones de cada componente
restrictions = [(-500, 500)]
# Instancia del algoritmo
abc = ArtificialBeeColony(5, 5, 50, f, 42)
# Ejecución del algoritmo
best_sol, best_val, history = abc.execute(restrictions)
# Representación gráfica del algoritmo
show_abc(history, abc.f, restrictions)
